In [13]:
# === Imports (один раз в начале ноутбука) ===
import sys, torch
sys.path.append("/mnt/data")
import sys, importlib
from snn_mnist_net import Cfg, run_experiment, save_snn, load_weights_into, build_net,build_encoder_from_cfg
from label_map import build_label_map,save_label_map, load_label_map
from counts_readout import collect_counts_plus_cuda, make_mnist_datasets
from readout_models import tfidf_from_counts, train_mlp_readout
# (опционально) from evaluation import evaluate_on_mnist

_MODS = [
    "snn_mnist_net",
    "label_map",
    "counts_readout",
    "readout_models",
]

# 1) вычистим из sys.modules сам модуль и подпакеты
for m in list(sys.modules):
    if m in _MODS or any(m.startswith(x + ".") for x in _MODS):
        sys.modules.pop(m, None)

# 2) инвалидируем кэши файловой системы
importlib.invalidate_caches()

# 3) заново импортируем модули как модули
import snn_mnist_net, label_map, counts_readout, readout_models

In [14]:


# === FULL RUN CONFIG (адаптировано под Cfg) ===
cfg = Cfg(
    # базовые
    time=200, n_hidden=100, device="cpu", seed=42,
    # обучение
    N=60000,                 # стартовый «полный» прогон
    log_every=500,
    # STDP
    nu_plus=1e-4, nu_minus=-1e-3,
    # ингибиция / WTA
    enable_inhibition_at_start=True,
    top_k=3,
    inhib_strength=0.705,
    # веса
    w_init_lo=0.25, w_init_hi=0.80,
    w_clip_min=0.0, w_clip_max=1.0,
    # пороги/динамика LIF
    thresh_init=0.38, tau_val=150.0, refrac_val=2.0, reset_val=0.0,
    # кодировщик
    encoder="poisson", poisson_rate_scale=0.011,
    # прочее
    debug=False,
    warmup_N = 100
    
)
print("FULL RUN:", {k: getattr(cfg, k) for k in [
    "inhib_strength","w_init_lo","w_init_hi","poisson_rate_scale","thresh_init","N"
]})




FULL RUN: {'inhib_strength': 0.705, 'w_init_lo': 0.25, 'w_init_hi': 0.8, 'poisson_rate_scale': 0.011, 'thresh_init': 0.38, 'N': 60000}


In [ ]:
import importlib, snn_mnist_net
importlib.reload(snn_mnist_net)
# === 1) Обучение SNN (STDP) ===
summary, FF, lif_layer, net, encoder = run_experiment(cfg, verbose=True, progress=True)
print("SUMMARY:", summary)

#save_snn("out/snn_mnist_v5.pt", cfg, FF, lif_layer)

In [ ]:
print("SUMMARY:", summary)

In [15]:
import os
import importlib
import evaluation 
from evaluation import probe_readouts_counts
importlib.invalidate_caches()
importlib.reload(snn_mnist_net)
importlib.reload(evaluation)
importlib.invalidate_caches()
net, input_layer, lif_layer, connection, recurrent_inh, _ = build_net(cfg)
enc = build_encoder_from_cfg(cfg)

# 2) загрузить веса и пороги
load_weights_into(net, connection, lif_layer, "out/snn_mnist_v5.pt")



[lif] thresh = tensor mean=0.380 std=0.000 shape=(100,)
[lif] tc_decay = scalar 150.000
[lif] refrac = scalar 2.000
[lif] reset = scalar 0.000
[lif] dt = scalar 1.000
Loaded from out/snn_mnist_v5.pt


In [ ]:
# 3) теперь вызываем build_label_map
label_map_build = build_label_map(net, None, lif_layer, enc, n_calib=2000, T=cfg.time, top_k=3, seed=123)

save_label_map("out/label_map_v5.pt",label_map_build, meta={"ckpt":"out/snn_mnist_v5.pt","T":cfg.time})

In [4]:
label_map_build = load_label_map("out/label_map_v5.pt")

[label_map] loaded: out/label_map_v5.pt  shape=(100,)  meta={'created_at': '2026-01-26 12:18:18', 'ckpt': 'out/snn_mnist_v5.pt', 'T': 200}


In [8]:
from counts_readout import make_mnist_datasets

ds_train, ds_test = make_mnist_datasets()

In [10]:
import importlib

from counts_readout import collect_counts_plus_fast, make_mnist_datasets
from evaluation import probe_readouts_counts, train_mlp_readout
importlib.invalidate_caches()
importlib.reload(counts_readout)
Xtr, ytr, _ = collect_counts_plus_fast(
    net, lif_layer, enc, ds_train, 60000, T=cfg.time, label_map=label_map_build,
    move_net=True, batch_size=128,
    encoder_rate_boost=5.5,          # было 3.0 → 1.5
    return_debug=True, desc="Collect train: boost=1.5"
)
Xte, yte, _ = collect_counts_plus_fast(
    net, lif_layer, enc, ds_test, 10000, T=cfg.time, label_map=label_map_build,
    move_net=True, batch_size=128,
    encoder_rate_boost=5.5,
    return_debug=True, desc="Collect test: boost=1.5"
)

N = lif_layer.n
print("mean sum per sample:", float(Xtr[:, :N].sum(1).mean()), float(Xte[:, :N].sum(1).mean()))

[collect_counts_plus_cuda] offset summary: requested=+0.000, applied=0, avgΔvt=0.000, restore_ok=–


[collect_counts_plus_cuda] offset summary: requested=+0.000, applied=0, avgΔvt=0.000, restore_ok=–
mean sum per sample: 414.9049377441406 423.0065002441406


In [11]:
N = lif_layer.n
Xtr_n, Xte_n, *_ = tfidf_from_counts(Xtr[:, :N], Xte[:, :N])
_, acc = train_mlp_readout(Xtr_n, ytr, Xte_n, yte, hidden=256, epochs=30, batch_size=256)
print(f"[MLP/TF-IDF] acc = {acc:.3f}")

[MLP/TF-IDF] acc = 0.544


In [7]:
print(dbg_tr)

{'requested_offset': 0.0, 'threshold_abs': None, 'encoder_rate_boost': 3.0, 'temp_disable_inh': False, 'applied_count': 0, 'avg_delta_mean_vt': 0.0, 'restore_ok_fraction': 1.0, 'vt_before_samples': [], 'vt_after_apply_samples': [], 'vt_after_restore_samples': [], 'device': 'cuda', 'batch_size': 64, 'write_pos': 60000, 'expected_batches': 938, 'actual_batches': 938, 'version': 3}


In [ ]:
print("train offset debug:", dbg_tr)

In [ ]:
print(f"[MLP / TF-IDF(counts)] test acc = {acc:.3f}")

In [ ]:
from label_map import load_label_map 
label_map_build = load_label_map ("out/label_map_v5.pt")

In [ ]:
import os
net, input_layer, lif_layer, connection, recurrent_inh, _ = build_net(cfg)
load_weights_into(net, connection, lif_layer, "out/snn_mnist_v5.pt")

In [16]:
from evaluation import eval_readouts_from_net
enc = build_encoder_from_cfg(cfg)
accs = eval_readouts_from_net(net, lif_layer, enc, cfg, label_map=label_map_build)
print(accs)   # ждём, что хотя бы counts_zscore+Linear или PCAwhiten+Linear > 0.6

[collect_counts_plus_cuda] offset summary: requested=+0.000, applied=0, avgΔvt=0.000, restore_ok=–


[collect_counts_plus_cuda] offset summary: requested=+0.000, applied=0, avgΔvt=0.000, restore_ok=–
{'counts_zscore+Linear': 0.4345000088214874, 'PCAwhiten+Linear': 0.42890000343322754, 'TFIDF+MLP': 0.5467000007629395}


In [ ]:
# базовая вариативность counts (только скрытые нейроны, без 10 последних)
N = lif_layer.n
Xtr_c = Xtr[:, :N]; Xte_c = Xte[:, :N]

print("Xtr counts>0 frac:", float((Xtr_c>0).float().mean()))
print("Xte counts>0 frac:", float((Xte_c>0).float().mean()))
print("mean sum per sample (train/test):",
      float(Xtr_c.sum(1).mean()), float(Xte_c.sum(1).mean()))
print("per-feature std (train) mean±std:",
      float(Xtr_c.std(0).mean()), float(Xtr_c.std(0).std()))

# топ-нейроны по дисперсии — должны быть >0
v = Xtr_c.std(0)
topv, idx = torch.topk(v, 5)
print("top var neurons:", idx.tolist(), [float(x) for x in topv])

In [ ]:


# Базовый конфиг — твои текущие дефолты
base = Cfg(
    time=200, n_hidden=100, encoder="poisson",
    enable_inhibition_at_start=True, top_k=3,
    nu_plus=1e-4, nu_minus=-1e-3,
    poisson_rate_scale=0.010, thresh_init=0.35,
    w_init_lo=0.50, w_init_hi=0.80,
    w_clip_min=0.0, w_clip_max=1.0,
    vt_mean=0.35, vt_jitter=0.02,
    tau_val=150.0, refrac_val=2.0,
    device="cpu", seed=42, debug=False,
    N=400
)

# Плотные значения — без добавления новых гиперов
grid_refine_ultra = {
    "inhib_strength":      [0.695, 0.700, 0.705],     # ±0.005
    "w_init_pairs":        [(0.25, 0.80)],            # фиксируем пару, она лучшая в твоих топах
    "poisson_rate_scale":  [0.0100, 0.0105, 0.0110],  # вниз на шаги 0.0005–0.001
    "thresh_init":         [0.34, 0.36, 0.38],        # проверить чуть ниже/выше 0.36
    "N_train":             400
}
_ = grid_search_network_v2(
    base_cfg=base,
    grid=grid_refine_ultra,
    csv_path="grid_refine_ultra.csv",
    max_combos=999,
    sample_mode="full",
    resume=True
)


In [ ]:
cands = [
    dict(name="A_balanced_hi_inh", inhib_strength=0.70, w_init_lo=0.25, w_init_hi=0.80, poisson_rate_scale=0.012, thresh_init=0.36),
    dict(name="B_balanced_mid",    inhib_strength=0.70, w_init_lo=0.45, w_init_hi=0.70, poisson_rate_scale=0.011, thresh_init=0.34),
    dict(name="C_balanced_low_inh",inhib_strength=0.50, w_init_lo=0.25, w_init_hi=0.80, poisson_rate_scale=0.011, thresh_init=0.42),
]

for cfg_over in cands:
    cfg2 = Cfg(
        time=200, n_hidden=100, encoder="poisson",
        enable_inhibition_at_start=True, top_k=3,
        nu_plus=1e-4, nu_minus=-1e-3,
        vt_mean=0.35, vt_jitter=0.02, tau_val=150.0, refrac_val=2.0,
        w_clip_min=0.0, w_clip_max=1.0,
        N=2000,  # увеличим окна обучения для проверки устойчивости
        device="cpu", seed=42, debug=False,
        # перезаписываем ключевые гиперы:
        inhib_strength=cfg_over["inhib_strength"],
        w_init_lo=cfg_over["w_init_lo"], w_init_hi=cfg_over["w_init_hi"],
        poisson_rate_scale=cfg_over["poisson_rate_scale"],
        thresh_init=cfg_over["thresh_init"],
    )
    print("\n=== TRY:", cfg_over["name"], "===\n")
    out, connection, lif_layer, net, encoder = run_experiment(cfg2, verbose=True, progress=True)
    


0.65: SUMMARY: {'spikes_per_sample': 81.135, 'winners_unique': 80, 'winner_HHI': 0.016851851851851854, 'energy_proxy_per_sample': 782.995}
BEFORE WarmUp removed: SUMMARY: {'spikes_per_sample': 81.135, 'winners_unique': 80, 'winner_HHI': 0.016851851851851854, 'energy_proxy_per_sample': 782.995}
WMIN -1.0: SUMMARY: {'spikes_per_sample': 76.391, 'winners_unique': 82, 'winner_HHI': 0.016152963242910327, 'energy_proxy_per_sample': 778.251}


Используем: cpu
[lif] thresh = tensor mean=0.350 std=0.000 shape=(100,)
[lif] tc_decay = scalar 150.000
[lif] refrac = scalar 2.000
[lif] reset = scalar 0.000
[lif] dt = scalar 1.000
Saved to out/snn_mnist.pt | W (784, 100)

SUMMARY: {'spikes_per_sample': 61.598, 'winners_unique': 82, 'winner_HHI': 0.022694477239931787, 'energy_proxy_per_sample': 765.368}

Используем: cpu

SNN-MNIST TRAIN START
seed=42  device=cpu  T=200  N=1000
n_hidden=100  thresh_init=0.35
STDP: nu_plus=0.0001  nu_minus=-0.001
FF init: w∈[0.8, 1.2], clip=[0.0,1.5]
Encoder=poisson  poisson_rate_scale=0.006  base_seed=123
Inhibition: enable=True  inhib_strength=0.65  top_k=3


[lif] thresh = tensor mean=0.350 std=0.000 shape=(100,)
[lif] tc_decay = scalar 150.000
[lif] refrac = scalar 2.000
[lif] reset = scalar 0.000
[lif] dt = scalar 1.000
Train (N=1000, T=200): 100%|████████████████████| 1000/1000 [02:42<00:00,  6.16it/s, in=122, lif=10]
Saved to out/snn_mnist.pt | W (784, 100)

SUMMARY: {'spikes_per_sample': 61.598, 'winners_unique': 82, 'winner_HHI': 0.022694477239931787, 'energy_proxy_per_sample': 765.368



SNN-MNIST TRAIN START
seed=42  device=cpu  T=200  N=1000
n_hidden=100  thresh_init=0.35
STDP: nu_plus=0.0001  nu_minus=-0.001
FF init: w∈[0.5, 0.8], clip=[0.0,1.0]
Encoder=poisson  poisson_rate_scale=0.01  base_seed=123
Inhibition: enable=True  inhib_strength=0.55  top_k=3

[lif] thresh = tensor mean=0.350 std=0.000 shape=(100,)
[lif] tc_decay = scalar 150.000
[lif] refrac = scalar 2.000
[lif] reset = scalar 0.000
[lif] dt = scalar 1.000
Train (N=1000, T=200): 100%|█████████████████████| 1000/1000 [02:49<00:00,  5.92it/s, in=192, lif=4]
Saved to out/snn_mnist.pt | W (784, 100)

SUMMARY: {'spikes_per_sample': 62.249, 'winners_unique': 92, 'winner_HHI': 0.02255912273748183, 'energy_proxy_per_sample': 1167.019}


In [ ]:
#res, connection, lif_layer, trained, encoder = run_experiment(cfg, verbose=True)   # тут ты уже обучал


# 2) сохраняем после обучения:
#save_snn("out/snn_mnist.pt", cfg, connection, lif_layer)

# 3) для оценки — пересобираем сеть (или используем текущую), грузим веса:
net, input_layer, lif_layer, connection, recurrent_inh = build_net(cfg)

load_weights_into(net, connection, lif_layer, "out/snn_mnist.pt")
label_map = build_label_map(net, input_layer, lif_layer, encoder,
                            n_calib=20000, T=cfg.time, top_k=cfg.top_k)
np.save("out/label_map.npy", label_map)

In [ ]:
from bindsnet.learning import PostPre
net, input_layer, lif_layer, connection, recurrent_inh = build_net(cfg)
# ONE CHANGE: вернуть v_thresh/thresh из сохранённой модели
load_weights_into(net, connection, lif_layer, "out/snn_mnist_v2.pt")  # это вернёт W и vt из ckpt
encoder = build_encoder_from_cfg(cfg)
# дальше как есть (ингибиция уже включена у тебя):
label_map = np.load("out/label_map_v2.npy")
#label_map = build_label_map(net, input_layer, lif_layer, encoder,
#                            n_calib=20000, T=cfg.time, top_k=cfg.top_k)
#np.save("out/label_map.npy", label_map)

[LDA + NCM (cosine)] m=9, test acc = 0.293
[NCM cosine / PCA-whitened] test acc = 0.293
[MLP / TF-IDF(counts)] test acc = 0.311

In [ ]:
import torch
from torchvision import transforms
from bindsnet.datasets import MNIST  # или torchvision.datasets.MNIST, если используешь её
# from torchvision.datasets import MNIST  # <- раскомментируй вместо строки выше при необходимости

# --- предполагается, что класс PoissonEncoder уже определён и использует base_seed внутри ---
# class PoissonEncoder: ... (из твоей ячейки)
# ВАЖНО: base_seed задаётся в конструкторе и НЕ передаётся при вызове

# === 0) Настройка ===
T = 200
RATE = 0.006
encoder = PoissonEncoder(T=T, rate_scale=RATE, device="cpu")

transform = transforms.Compose([transforms.ToTensor()])
dataset = MNIST(root="./data", train=True, download=True, transform=transform)

# Возьмём несколько индексов с непустыми изображениями (чтобы были не нулевые спайки)
nonzero_idxs = []
for i in range(0, len(dataset), 113):  # разреженный проход
    img = dataset[i]["image"]
    if float(img.sum()) > 0.0:
        nonzero_idxs.append(i)
    if len(nonzero_idxs) >= 10:
        break

assert len(nonzero_idxs) > 0, "Не нашёл непустых картинок в выборке — проверь transform/датасет."

# === 1) Один и тот же образ -> тот же спайк-поезд (строгая равенство по элементам) ===
for i in nonzero_idxs:
    x = dataset[i]["image"]

    s1 = encoder(x)          # первый вызов
    _ = torch.rand(7)        # «шумим» внешний RNG — не должно влиять
    s2 = encoder(x)          # второй вызов

    same = bool((s1 == s2).all())
    print(f"[SAME IMG] idx={i:5d} equal={same}  spikes_sum={int(s1.sum())}")
    assert same, f"Кодирование не детерминировано для idx={i}"

print("✓ Один и тот же образ всегда кодируется одинаково (детерминированно).")

# === 2) Разные образы -> разные спайки (для не-пустых) ===
# (теоретически два разных изображения могут дать совпадающие спайки при очень малом RATE,
#  поэтому считаем успехом, если для большинства пар есть различия)
diff_count = 0
pairs_checked = 0
for a, b in zip(nonzero_idxs, nonzero_idxs[1:]):
    xa, xb = dataset[a]["image"], dataset[b]["image"]
    sa, sb = encoder(xa), encoder(xb)
    different = bool((sa != sb).any())
    pairs_checked += 1
    diff_count += int(different)
    print(f"[DIFF IMG] {a:5d} vs {b:5d} different={different}")

if pairs_checked > 0:
    ratio = diff_count / pairs_checked
    print(f"Различимых пар: {diff_count}/{pairs_checked} ({ratio:.0%})")
    # Не делаем assert на 100%, т.к. при очень маленьком RATE возможны редкие совпадения
    assert ratio >= 0.7, "Слишком мало различий между разными картинками — проверь RATE/реализацию."

print("✓ Тест PoissonEncoder пройден.")
